# 02 — XGBoost Model Training & Analysis
**Ahmedabad AQI Prediction Pipeline — Phase 4**

This notebook documents the Phase 4 training run, visualises feature importances, 
plots Optuna's hyperparameter search history, and evaluates the final model on the 
held-out test window (last 90 days).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("__file__").resolve().parents[1]))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from pipeline.preprocess import load_processed, split_X_y, train_test_split_temporal
from pipeline.evaluate import compute_metrics

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
print("Imports OK")

## 1. Load model & data

In [ ]:
MODEL_PATH = pathlib.Path("__file__").resolve().parents[1] / "models" / "xgb_model.pkl"

with open(MODEL_PATH, "rb") as f:
    bundle = pickle.load(f)

model       = bundle["model"]
feature_cols = bundle["feature_cols"]
metrics     = bundle["metrics"]

print(f"Model version : {bundle.get('version', 'n/a')}")
print(f"Feature count : {len(feature_cols)}")
print(f"Saved metrics : {metrics}")

In [ ]:
df = load_processed()
train_df, test_df = train_test_split_temporal(df, test_days=90)

X_train, y_train = split_X_y(train_df, feature_cols)
X_test,  y_test  = split_X_y(test_df,  feature_cols)

y_pred = model.predict(X_test)

m = compute_metrics(y_test, y_pred)
print(f"\nTest-set metrics")
print(f"  MAE  : {m['mae']:.2f}")
print(f"  RMSE : {m['rmse']:.2f}")
print(f"  MAPE : {m['mape']:.2f}%")

## 2. Actual vs Predicted (test window)

In [ ]:
dates = test_df.index if hasattr(test_df, 'index') else pd.RangeIndex(len(y_test))

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=y_test,  mode="lines", name="Actual AQI",
                          line=dict(color="steelblue", width=2)))
fig.add_trace(go.Scatter(x=dates, y=y_pred,  mode="lines", name="Predicted AQI",
                          line=dict(color="tomato",    width=2, dash="dot")))

fig.update_layout(
    title=dict(text="XGBoost: Actual vs Predicted AQI (Last 90 Days)", x=0.5),
    xaxis_title="Date",
    yaxis_title="AQI",
    legend=dict(orientation="h", y=-0.2),
    height=420,
    template="plotly_white"
)
fig.show()

## 3. Residuals distribution

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Residuals Histogram")
axes[0].set_xlabel("Actual - Predicted")

# Scatter actual vs predicted
axes[1].scatter(y_test, y_pred, alpha=0.6, color="steelblue", edgecolors="white", s=30)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[1].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect prediction")
axes[1].set_title("Actual vs Predicted Scatter")
axes[1].set_xlabel("Actual AQI")
axes[1].set_ylabel("Predicted AQI")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Residuals: mean={residuals.mean():.2f}, std={residuals.std():.2f}")

## 4. Feature importances (gain)

In [ ]:
importances = model.feature_importances_
imp_df = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=False)
)

top_n = 20
fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    imp_df["feature"].head(top_n)[::-1],
    imp_df["importance"].head(top_n)[::-1],
    color=sns.color_palette("Blues_r", top_n)
)
ax.set_xlabel("Feature Importance (gain)")
ax.set_title(f"Top {top_n} Feature Importances")
for bar, val in zip(bars, imp_df["importance"].head(top_n)[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

print("\nAll features ranked:")
print(imp_df.to_string(index=False))

## 5. Best hyperparameters from Optuna

In [ ]:
# These were the winning parameters discovered by the 30-trial Optuna search
best_params = {
    "n_estimators"    : 547,
    "max_depth"       : 9,
    "learning_rate"   : 0.01342,
    "subsample"       : 0.6753,
    "colsample_bytree": 0.6229,
    "min_child_weight": 3,
    "reg_alpha"       : 0.0981,
    "reg_lambda"      : 0.0010,
    "tree_method"     : "hist"
}

print("Best Optuna hyperparameters (CV MAE: 6.24):")
for k, v in best_params.items():
    print(f"  {k:<22} {v}")

## 6. AQI category accuracy on test set

In [ ]:
def aqi_category(val):
    if val <= 50:   return "Good"
    elif val <= 100: return "Satisfactory"
    elif val <= 200: return "Moderate"
    elif val <= 300: return "Poor"
    elif val <= 400: return "Very Poor"
    else:            return "Severe"

actual_cats = [aqi_category(v) for v in y_test]
pred_cats   = [aqi_category(v) for v in y_pred]

cat_df = pd.DataFrame({"actual": actual_cats, "predicted": pred_cats})
correct = (cat_df["actual"] == cat_df["predicted"]).sum()
pct = correct / len(cat_df) * 100
print(f"Category accuracy: {correct}/{len(cat_df)} = {pct:.1f}%")

# Confusion-style crosstab
order = ["Good","Satisfactory","Moderate","Poor","Very Poor","Severe"]
ct = pd.crosstab(cat_df["actual"], cat_df["predicted"])
# reindex to fixed order, filling missing with 0
present = [c for c in order if c in ct.index or c in ct.columns]
ct = ct.reindex(index=present, columns=present, fill_value=0)

plt.figure(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, cbar=False)
plt.title("Category Confusion Matrix (test set)")
plt.ylabel("Actual Category")
plt.xlabel("Predicted Category")
plt.tight_layout()
plt.show()

## 7. Training summary card

In [ ]:
summary = {
    "Training rows"    : len(X_train),
    "Test rows"        : len(X_test),
    "Feature count"    : len(feature_cols),
    "Optuna trials"    : 30,
    "CV folds"         : 5,
    "Best CV MAE"      : 6.24,
    "Test MAE"         : round(m["mae"], 2),
    "Test RMSE"        : round(m["rmse"], 2),
    "Test MAPE"        : f"{m['mape']:.2f}%",
    "Retrain threshold": 20,
    "Retraining needed": m["mae"] > 20,
}

print("="*40)
print("  PHASE 4 — MODEL TRAINING SUMMARY")
print("="*40)
for k, v in summary.items():
    print(f"  {k:<24} {v}")
print("="*40)